<a href="https://colab.research.google.com/github/ZarmelZar/ZarmelZar/blob/main/Project_Comp_215_Melika_Mousavikhah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Driving Directions Tool:**


This project presents a Python-based solution to find and display driving directions between two locations. The application takes **street addresses** as input, converts them into **geographic coordinates**, and then plots the **route** on a map. The system uses available APIs and open-source libraries to show real-world geographic data processing and mapping. The project combines several libraries and APIs to load geographic coordinates, calculate routes, and display maps.


Here's the modules that are used for the following code:
1. **requests** for API communication
2. **openrouteservice** for route data
3. **folium** for map rendering
4. **polyline** for decoding



In [18]:
!pip install openrouteservice folium
!pip install polyline
import requests
import folium
from openrouteservice import convert
import polyline

# **Data Loader Class**

The ***DataLoader*** class is responsible for obtaining coordinates and loading driving routes between two locations. It uses the **Nominatim API** from **OpenStreetMap** to convert street addresses into geographic coordinates, and the ***get_user_location()*** method is responsible for asking the user to input two street addresses. These addresses are then converted into coordinates, and the ***get_coordinates()*** method is used to load the **Nominatim API** and retrieve the coordinates from the response. The class also uses the **OpenRouteService Directions API** to calculate the route between two geographic coordinates, with the ***get_route()*** method sending the start and end coordinates to the API, receiving the route data, and decoding the route geometry using **polyline**.

In [19]:
class DataLoader:

    def __init__(self):
        self.geolocator_url = "https://nominatim.openstreetmap.org/search"
        self.api_url = "https://api.openrouteservice.org/v2/directions/driving-car"
        self.api_key = "5b3ce3597851110001cf62486fa8e28a936a46708e15a25f10f55adc"

    def get_user_location(self):
        start_address = input("What is your current location address?: ")
        start_coords = self.get_coordinates(start_address)

        destination_address = input("What is the destination address?: ")
        destination_coords = self.get_coordinates(destination_address)

        if start_coords and destination_coords:
            return start_coords, destination_coords
        else:
            print("Could not find coordinates for the given addresses.")
            return None, None

    def get_coordinates(self, location_name):
        params_dict = {'q': location_name, 'format': 'json'}
        headers = {"User-Agent": "MyPathfinderApp/1.0 (Melikamsk122@gmail.com)"}

        response = requests.get(self.geolocator_url, params=params_dict, headers=headers)

        if response.status_code == 200:
            try:
                data = response.json()

                if len(data) > 0:
                    coords = [data[0]['lon'], data[0]['lat']]
                    print(f"Latitude: {coords[1]}, Longitude: {coords[0]}")
                    return {'latitude': coords[1], 'longitude': coords[0]}
                else:
                    print(f"No results found for {location_name}.")
                    return None
            except ValueError as e:
                print(f"Error decoding JSON: {e}")
                print(f"Response text: {response.text}")
                return None
        else:
            print(f"Error loading data from OpenStreetMap: {response.status_code}")
            return None

    def get_route(self, start_coords, end_coords):
        headers = {'Authorization': self.api_key, 'Content-Type': 'application/json'}

        body = {"coordinates": [[start_coords['longitude'], start_coords['latitude']],[end_coords['longitude'], end_coords['latitude']]]}

        response = requests.post(self.api_url, json=body, headers=headers)


        if response.status_code == 200:
            try:
                data = response.json()

                if 'routes' in data and len(data['routes']) > 0:
                    route = []
                    geometry = data['routes'][0]['geometry']
                    decoded_route = polyline.decode(geometry)

                    for segment in data['routes'][0]['segments']:
                        for step in segment['steps']:
                            route.append((step['instruction'], step['distance'], step['duration']))


                    return route, decoded_route
                else:
                    print("No route found.")
                    return None, None
            except ValueError as e:
                print(f"Error decoding JSON: {e}")
                print(f"Response text: {response.text}")
                return None, None
        else:
            print(f"Error loading route data: {response.status_code}")
            return None, None


The ***format_route_instructions()*** function takes a list of route steps (instruction, distance, duration) and returns a string of just the **instructions**.



*If the route is empty, it returns a message saying no instructions are available.

In [20]:
def format_route_instructions(route):
    """
    Takes a list of route steps in the format (instruction, distance, duration)
    and returns a formatted string of just the instructions.
    """
    if not route:
        return "No route instructions available."

    formatted_instructions = "Suggested Route:\n"
    for step in route:
        instruction = step[0]
        formatted_instructions += f"{instruction}\n"

    return formatted_instructions

## MapVisualizer class

The ***MapVisualizer class*** is responsible for creating and displaying an interactive map with the given start and destination coordinates, as well as the calculated route. It initializes with the start and destination coordinates, the route, and the decoded route. The ***create_map*** method creates a map centered on the starting point, adds markers for the start and destination locations, and displays the route as a **polyline** on the map. Finally, it displays the interactive map using **Folium**.

In [21]:
class MapVisualizer:
    def __init__(self, start_coords, destination_coords, route, decoded_route):
        self.start_coords = start_coords
        self.destination_coords = destination_coords
        self.route = route
        self.decoded_route = decoded_route

    def create_map(self):
        m = folium.Map(location=[self.start_coords['latitude'], self.start_coords['longitude']], zoom_start=14)

        folium.Marker([self.start_coords['latitude'], self.start_coords['longitude']], popup="Start").add_to(m)

        folium.Marker([self.destination_coords['latitude'], self.destination_coords['longitude']], popup="Destination").add_to(m)

        folium.PolyLine(self.decoded_route, color="blue", weight=2.5, opacity=1).add_to(m)

        display(m)

# Running the Route and Map Visualization
This part of the code performs the following:

1. It creates an instance of the DataLoader class (dl).

2. It calls get_user_location() to get the start and destination coordinates from the user.

3. If valid coordinates are found, it retrieves the route and decoded route using get_route(), then formats and prints the route instructions with format_route_instructions().

4. If a valid route is found, it creates an instance of the MapVisualizer class and displays the route on the map.

5. If no route is found, it prints a message indicating that no route could be displayed.

6. If coordinates cannot be found, it prints an error message.




*-To see the map, open Colab and run the code.*


In [23]:
dl = DataLoader()

start_coords, destination_coords = dl.get_user_location()

if start_coords and destination_coords:
    route, decoded_route = dl.get_route(start_coords, destination_coords)
    formatted = format_route_instructions(route)
    print(formatted)
    if route and decoded_route:
        map_visualizer = MapVisualizer(start_coords, destination_coords, route, decoded_route)
        map_visualizer.create_map()
    else:
        print("No route found to display on map.")
else:
    print("Could not find coordinates for the given addresses.")
answer = input("Do you want to see the Route Dist?:")
if answer == 'Yes':
  print(route)
else:
  print("Thank you for using our app!")

What is your current location address?: 2055 Purcell Way North Vancouver
Latitude: 49.317583189725674, Longitude: -123.02098513864625
What is the destination address?: 2008 Fullerton Avenue North VAncouver
Latitude: 49.330807199999995, Longitude: -123.12104216138323
Suggested Route:
Head northwest on Purcell Way
Turn left onto Lillooet Road
Keep right
Turn right onto Mount Seymour Parkway
Keep right
Keep left
Keep right
Keep left
Turn left onto Taylor Way, 99, 1A
Turn left onto Inglewood Avenue
Turn sharp left onto Keith Road
Arrive at Keith Road, on the right



Do you want to see the Route Dist?:Yes
[('Head northwest on Purcell Way', 220.8, 53.0), ('Turn left onto Lillooet Road', 550.7, 85.4), ('Keep right', 55.2, 9.9), ('Turn right onto Mount Seymour Parkway', 76.0, 10.9), ('Keep right', 585.7, 53.2), ('Keep left', 7706.3, 541.4), ('Keep right', 338.4, 24.4), ('Keep left', 451.4, 49.1), ('Turn left onto Taylor Way, 99, 1A', 400.1, 37.3), ('Turn left onto Inglewood Avenue', 1174.7, 117.7), ('Turn sharp left onto Keith Road', 202.1, 48.5), ('Arrive at Keith Road, on the right', 0.0, 0.0)]


# Conclusion
This code effectively demonstrates how to take user inputs, calculate driving routes using external APIs, and display them on an interactive map. By using the Nominatim and OpenRouteService APIs, users can easily view the route and driving instructions.